In [8]:
!pip install langchain langchain-community langchain-openai chromadb pypdf sentence-transformers

     |████████████████████████████████| 75 kB 1.6 MB/s eta 0:00:011
     |████████████████████████████████| 39.4 MB 532 kB/s eta 0:00:01     |██████████████████████████▋     | 32.7 MB 302 kB/s eta 0:00:22
  Attempting uninstall: scipy
    Found existing installation: scipy 1.7.3
    Uninstalling scipy-1.7.3:
      Successfully uninstalled scipy-1.7.3


In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.chains import RetrievalQA

PDF_FILE = "sample.pdf"

def load_documents(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(documents)
    return chunks


def build_vector_store(documents):

    embeddings = OpenAIEmbeddings()

    vector_store = Chroma.from_documents(
        documents,
        embedding=embeddings,
        persist_directory="./vector_db"
    )

    vector_store.persist()

    return vector_store


def create_rag_system(vector_store):

    retriever = vector_store.as_retriever(search_kwargs={"k": 3})

    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0
    )

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever
    )

    return qa_chain


documents = load_documents(PDF_FILE)

vector_store = build_vector_store(documents)

qa_system = create_rag_system(vector_store)

while True:
    question = input("Ask a question (type exit to quit): ")

    if question.lower() == "exit":
        break

    answer = qa_system.run(question)

    print("\nAnswer:", answer)